In [ ]:
import ROOT, glob, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"
dataPath = "/dpnc/beegfs/dampe/tryf/ForKimberly/"
exposurePath = "/dpnc/beegfs/dampe/tryf/ForKimberly/exposureMaps_GIE/"
exponent = 0

In [ ]:
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import EarthLocation

LEAP_SECONDS = [
    "2015-07-01T00:00:00",  # +1 leap second
    "2017-01-01T00:00:00",  # +1 leap second
]

def add_leap_seconds(mjd1, mjd2):
    times = Time(mjd1, mjd2, scale='tt', format='mjd')

    total_correction = np.zeros(len(times))  # seconds

    for leap in LEAP_SECONDS:
        time_leap = Time(leap, scale='tt')
        total_correction += (times > time_leap).astype(int) # Adds 1 or 0 -> Plays on boolean type

    corrected_times = times + TimeDelta(total_correction, format='sec')

    mjd1_new = corrected_times.tt.jd1 - 2400000.5
    mjd2_new = corrected_times.tt.jd2

    return mjd1_new, mjd2_new

def met_to_mjd_tt_split(seconds, milliseconds):
    # Total delta since epoch: seconds + milliseconds, kept separate
    seconds = np.asarray(seconds)
    milliseconds = np.asarray(milliseconds)

    # Construct TimeDelta in two parts to preserve precision
    delta_sec = TimeDelta(seconds, format='sec')
    delta_ms  = TimeDelta(milliseconds * 1e-3, format='sec')  # convert ms to s

    # MET epoch in TT 
    epoch = Time("2013-01-01T00:00:00", scale="utc")

    full_time = epoch + delta_sec + delta_ms

    # Get split MJD
    mjd1 = full_time.tt.jd1 - 2400000.5
    mjd2 = full_time.tt.jd2

    return mjd1, mjd2

def ecef_to_eci(mjd1, mjd2, x, y, z):
    t = Time(mjd1, mjd2, scale='tt', format='mjd')

    if isinstance(x, pd.Series):
        x = x.to_numpy() * u.m
    if isinstance(y, pd.Series):
        y = y.to_numpy() * u.m
    if isinstance(z, pd.Series):
        z = z.to_numpy() * u.m

    loc_ecef = EarthLocation(x=x, y=y, z=z)

    pos_gcrs, vel_gcrs = loc_ecef.get_gcrs_posvel(t)


    # TODO when we have sat velocities, figure out whether we can correct them too
    return pos_gcrs.get_xyz()




In [ ]:
dataDir = "/home/users/l/lebrun/data/"
dataName = "DmlNtup_Data_9years_FullPhotonData_AllSky.root"
tree = None
f = ROOT.TFile(dataDir + dataName, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))
rdf = ROOT.RDataFrame(tree)
df = rdf.AsNumpy(["celestialcoordBest_2", "celestialcoordBest_3", "tt_bgoTotalEcorr_GeV", 'tt_second', 'tt_milisecond'])
df = pd.DataFrame(df)

df = df.rename(columns={"celestialcoordBest_2" : "lon", "celestialcoordBest_3" : "lat", 
                        "tt_bgoTotalEcorr_GeV" : "energy"})

In [ ]:
dR = 10 # degrees, radius of region of interest
lon_target = 195.13
lat_target = 4.27 

lon_min = lon_target - dR
lon_max = lon_target + dR
lat_min = lat_target - dR
lat_max = lat_target + dR

df = df.loc[np.sqrt((np.rad2deg(df["lon"]) - lon_target)**2 + (np.rad2deg(df["lat"])-lat_target)**2) < dR]

In [ ]:
df["MJD1"], df["MJD2"] = met_to_mjd_tt_split(df["tt_second"], df["tt_milisecond"])
df["MJD1"], df["MJD2"] = add_leap_seconds(df["MJD1"].values, df["MJD2"].values)

# df['X_ECI'], df['Y_ECI'], df['Z_ECI'] = ecef_to_eci(df["MJD1"], df["MJD2"],
#                                                      df['tt_sat_position_x'],
#                                                      df['tt_sat_position_y'],
#                                                      df['tt_sat_position_z'])

print(f"Minimal day : {df['MJD1'].min()}, max day : {df['MJD1'].max()}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.time import Time

def fractional_year_to_iso(year_float):
    """
    Convert a fractional year (e.g., 2019.5) to ISO date string.
    """
    year = int(np.floor(year_float))
    remainder = year_float - year
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year+1}-01-01")
    delta = end - start
    result = start + pd.to_timedelta(remainder * delta)
    return result.isoformat()

def plot_histogram_calendar(df, start_year, end_year, bins=100, column="MJD1"):
    """
    Plot a histogram of data between fractional start_year and end_year.
    X-axis uses calendar dates for readability.

    Parameters:
    - df: pandas DataFrame with 'MJD1' column (or combined MJD).
    - start_year: float, start (e.g., 2019.5 for mid-2019)
    - end_year: float, end (e.g., 2020.25 for Q1 2020)
    - bins: int, number of histogram bins
    - column: str, column name (default="MJD1")
    """
    # Convert fractional years to ISO and then to MJD
    start_iso = fractional_year_to_iso(start_year)
    end_iso = fractional_year_to_iso(end_year)
    start_mjd = Time(start_iso, scale="tt").mjd
    end_mjd = Time(end_iso, scale="tt").mjd

    # Filter data
    mask = (df[column] >= start_mjd) & (df[column] < end_mjd)
    subset = df.loc[mask]

    if subset.empty:
        print(f"No data between {start_year} and {end_year}")
        return

    # Convert MJD to datetime for x-axis
    dates = Time(subset[column], format="mjd", scale="tt").to_datetime()

    # Plot
    plt.figure(figsize=(12, 5))
    plt.hist(dates, bins=bins, color='skyblue', edgecolor='black')
    plt.title(f"Histogram of {column} from {start_year} to {end_year}")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()


In [ ]:
# Mid-2019 to mid-2020, 200 bins for finer resolution
top = 2020.1
bot = 2019.5
plot_histogram_calendar(df, bot, top, bins=int((top-bot)*365))


In [ ]:
plot_histogram_calendar(df, 2016, 2025, bins=200)